In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [2]:
class Expert(nn.Module):
    def __init__(self, d_model, dim_feedforward, dropout):
        super().__init__()
        self.ffn = nn.Sequential(
            nn.Linear(d_model, dim_feedforward),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim_feedforward, d_model),
            nn.Dropout(dropout)
        )
    
    def forward(self, x):
        self.ffn(x)

In [3]:
class TopKRouter(nn.Module):
    def __init__(self, d_model, num_experts, top_k=2):
        super().__init__()
        self.num_experts = self.num_experts
        self.top_k = top_k
        self.gate = nn.Linear(d_model, num_experts)
        
    def forward(self, x):
        logits = self.gate(x)
        
        top_k_logits, top_k_indices = torch.topk(logits, self.top_k, dim=-1)
        
        top_k_probs = F.softmax(top_k_indices, dim=-1)
        
        return top_k_probs, top_k_indices